# Scrollie — SLM-SAM2 Viewer (Lambda)

Three panels per slice:
- **Left**: original fat-fraction image
- **Centre**: MuscleMap WB mask (prompt)
- **Right**: SLM-SAM2 propagated result

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ipywidgets'], check=True)
print('ipywidgets ready')

In [ ]:
import glob, io, os, re
import numpy as np
import matplotlib
matplotlib.use('Agg')          # no display needed — we render to bytes
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as _cm
import SimpleITK as sitk
import ipywidgets as widgets
from IPython.display import display

In [ ]:
SEG_DIR     = os.path.expanduser('~/slm_sam2_segs')
MM_SEGS_DIR = os.path.expanduser('~/MuscleMap_segs')
FATFRAC_DIR = os.path.expanduser(
    '~/myosegmenTUM/myosegmenTUM/myosegmenTUM/FATFRACTION_collected')

MUSCLE_NAMES = [
    'Vastus_Lateralis_L',   'Vastus_Lateralis_R',
    'Vastus_Intermedius_L', 'Vastus_Intermedius_R',
    'Vastus_Medialis_L',    'Vastus_Medialis_R',
    'Rectus_Femoris_L',     'Rectus_Femoris_R',
    'Sartorius_L',          'Sartorius_R',
    'Gracilis_L',           'Gracilis_R',
    'Semimembranosus_L',    'Semimembranosus_R',
    'Semitendinosus_L',     'Semitendinosus_R',
    'Biceps_Femoris_L',     'Biceps_Femoris_R',
    'Adductor_Magnus_L',    'Adductor_Magnus_R',
]

MM_LABEL_MAP = {
    7101: 'Vastus_Lateralis_L',   7102: 'Vastus_Lateralis_R',
    7111: 'Vastus_Intermedius_L', 7112: 'Vastus_Intermedius_R',
    7121: 'Vastus_Medialis_L',    7122: 'Vastus_Medialis_R',
    7131: 'Rectus_Femoris_L',     7132: 'Rectus_Femoris_R',
    7141: 'Sartorius_L',          7142: 'Sartorius_R',
    7151: 'Gracilis_L',           7152: 'Gracilis_R',
    7161: 'Semimembranosus_L',    7162: 'Semimembranosus_R',
    7171: 'Semitendinosus_L',     7172: 'Semitendinosus_R',
    7181: 'Biceps_Femoris_L',     7182: 'Biceps_Femoris_R',
    7201: 'Adductor_Magnus_L',    7202: 'Adductor_Magnus_R',
}

cmap = _cm.get_cmap('tab20', len(MUSCLE_NAMES))
name_to_color = {name: cmap(i) for i, name in enumerate(MUSCLE_NAMES)}

legend_patches = [
    mpatches.Patch(color=cmap(i), alpha=0.6, label=name)
    for i, name in enumerate(MUSCLE_NAMES)
]

npz_files    = sorted(glob.glob(os.path.join(SEG_DIR, '*_slmsam2.npz')))
file_options = {
    os.path.basename(p).replace('_slmsam2.npz', ''): p
    for p in npz_files
}

def stem_to_nii(stem):
    subject = stem.split('_FATFRACTION')[0]
    stack_n = re.search(r'stack(\d+)', stem).group(1)
    return os.path.join(FATFRAC_DIR, f'{subject}_FATFRACTION',
                        f'{subject}_FATFRACTION_stack{stack_n}.nii')

def stem_to_mm(stem):
    return os.path.join(MM_SEGS_DIR, f'{stem}_dseg.nii.gz')

print(f'Found {len(file_options)} SLM-SAM2 result files')

In [ ]:
def build_overlay_npz(npz_data, slice_idx):
    """RGBA overlay for SLM-SAM2 NPZ result at one slice."""
    h, w = next(iter(npz_data.values())).shape[1:]
    overlay = np.zeros((h, w, 4), dtype=float)
    for name in MUSCLE_NAMES:
        if name not in npz_data:
            continue
        mask = npz_data[name][slice_idx]
        if mask.any():
            c = name_to_color[name]
            overlay[mask == 1] = [c[0], c[1], c[2], 0.55]
    return overlay

def build_overlay_mm(mm_arr, slice_idx):
    """RGBA overlay for MuscleMap NIfTI label map at one slice."""
    sl = mm_arr[slice_idx]
    overlay = np.zeros((*sl.shape, 4), dtype=float)
    for label_idx, name in MM_LABEL_MAP.items():
        mask = (sl == label_idx)
        if mask.any():
            c = name_to_color[name]
            overlay[mask] = [c[0], c[1], c[2], 0.55]
    return overlay

def load_stack(label):
    npz_path = file_options[label]
    nii_path = stem_to_nii(label)
    mm_path  = stem_to_mm(label)

    img_arr = sitk.GetArrayFromImage(sitk.ReadImage(nii_path)).astype(float)
    img_norm = (img_arr - img_arr.min()) / (img_arr.max() - img_arr.min() + 1e-8)

    npz_data = dict(np.load(npz_path))

    if os.path.exists(mm_path):
        mm_sitk = sitk.ReadImage(mm_path)
        mm_arr  = sitk.GetArrayFromImage(mm_sitk).astype(np.int32)
    else:
        mm_arr = None

    return img_norm, npz_data, mm_arr

print('Helpers defined')

In [ ]:
def fig_to_png(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    return buf.read()

def render_slice(img_norm, npz_data, mm_arr, label, slice_idx):
    img = img_norm[slice_idx]
    n_panels = 3 if mm_arr is not None else 2
    fig, axes = plt.subplots(1, n_panels, figsize=(6 * n_panels, 6))

    axes[0].imshow(img, cmap='gray', origin='lower')
    axes[0].set_title(f'Image  (slice {slice_idx})')
    axes[0].axis('off')

    if mm_arr is not None:
        axes[1].imshow(img, cmap='gray', origin='lower')
        axes[1].imshow(build_overlay_mm(mm_arr, slice_idx), origin='lower')
        axes[1].set_title('MuscleMap WB (prompt)')
        axes[1].axis('off')
        ax_slm = axes[2]
    else:
        ax_slm = axes[1]

    ax_slm.imshow(img, cmap='gray', origin='lower')
    ax_slm.imshow(build_overlay_npz(npz_data, slice_idx), origin='lower')
    ax_slm.set_title('SLM-SAM2 result')
    ax_slm.axis('off')
    ax_slm.legend(handles=legend_patches, loc='lower right',
                  fontsize=5, framealpha=0.7, ncol=2)

    fig.suptitle(label, fontsize=9)
    plt.tight_layout()
    return fig_to_png(fig)

print('Render function defined')

In [ ]:
import base64

_cache       = {}
file_dropdown = widgets.Dropdown(options=list(file_options.keys()),
                                 description='Stack:')
slice_slider  = widgets.IntSlider(min=0, max=1, step=1, value=0,
                                  description='Slice:',
                                  layout=widgets.Layout(width='700px'))
html_widget   = widgets.HTML()
err_widget    = widgets.Label(value='')

def get_stack(label):
    if label not in _cache:
        _cache[label] = load_stack(label)
    return _cache[label]

def png_to_html(png_bytes):
    b64 = base64.b64encode(png_bytes).decode('ascii')
    return f'<img src="data:image/png;base64,{b64}" style="width:100%;max-width:1800px;"/>'

def update(label, slice_idx):
    try:
        img_norm, npz_data, mm_arr = get_stack(label)
        png = render_slice(img_norm, npz_data, mm_arr, label, slice_idx)
        html_widget.value = png_to_html(png)
        err_widget.value  = ''
    except Exception as e:
        import traceback
        err_widget.value = traceback.format_exc()

def on_file_change(change):
    label = change['new']
    try:
        img_norm, _, _ = get_stack(label)
        slice_slider.max   = img_norm.shape[0] - 1
        slice_slider.value = 0
        update(label, 0)
    except Exception as e:
        err_widget.value = str(e)

def on_slice_change(change):
    update(file_dropdown.value, change['new'])

file_dropdown.observe(on_file_change, names='value')
slice_slider.observe(on_slice_change, names='value')

display(widgets.VBox([file_dropdown, slice_slider, err_widget, html_widget]))

if file_options:
    label = file_dropdown.value
    img_norm, _, _ = get_stack(label)
    slice_slider.max = img_norm.shape[0] - 1
    update(label, 0)